In [27]:
# Construye el modelo
import pyomo.environ as pe

# Resuelve el modelo
import pyomo.opt as po

In [28]:
model = pe.ConcreteModel()

Sets

In [29]:
countries = [
    "Spain",
    "France",
    "Germany",
    "Austria",
    "Switzerland",
    "Italy"
]

model.car_types = pe.Set(initialize = ["A","B","C"])
model.countries = pe.Set(initialize = countries)

Parameters

In [30]:
cost_dict = {
    ("A", "Spain"): 160, ("A", "France"): 210, ("A", "Germany"): 180, ("A", "Austria"): 110, ("A", "Switzerland"): 85, ("A", "Italy"): 170,
    ("B", "Spain"): 120, ("B", "France"): 240, ("B", "Germany"): 165, ("B", "Austria"): 135, ("B", "Switzerland"): 100, ("B", "Italy"): 160,
    ("C", "Spain"): 150, ("C", "France"): 200, ("C", "Germany"): 175, ("C", "Austria"): 140, ("C", "Switzerland"): 115, ("C", "Italy"): 135
}

model.unitary_cost = pe.Param(model.car_types, model.countries, initialize=cost_dict)
model.change_cost = pe.Param(initialize=25)

In [31]:
# Países desde los que podemos cambiar de coche
model.change_countries = pe.Set(
    initialize=countries[:-1]
)

# País siguiente a cada país
next_country = dict(zip(countries[:-1], countries[1:]))

## Variables de decisión

In [32]:
model.x = pe.Var(model.countries, model.car_types, within=pe.Binary)
model.y = pe.Var(model.change_countries, within=pe.Binary)

## Función objetivo

Minimizar el coste del combustible más 25 € por cada cambio de tipo de coche.

In [ ]:
def obj_rule(model):
    fuel_cost = sum(
        model.unitary_cost[t, c] * model.x[c, t]
        for c in model.countries
        for t in model.car_types
    )
    switching_cost = sum(
        model.change_cost * model.y[c]
        for c in model.change_countries
    )
    return fuel_cost + switching_cost


model.total_cost = pe.Objective(rule=obj_rule, sense=pe.minimize)

## Restricciones

En cada país se elige exactamente un tipo de coche. Las dos desigualdades de cambio linealizan el valor absoluto de los apuntes: $|x_{c,t}-x_{c+1,t}|\le y_c$.

In [34]:
# Exactamente un tipo de coche en cada país.
def one_car_rule(model, c):
    return sum(model.x[c, t] for t in model.car_types) == 1


model.one_car = pe.Constraint(model.countries, rule=one_car_rule)

In [ ]:
# Si un tipo de coche deja de utilizarse entre dos países, y[c] debe valer 1.
def change_rule_1(model, c, t): # Determinar cuando se deja de usar el coche
    next_c = next_country[c]
    return model.x[c, t] - model.x[next_c, t] <= model.y[c]


model.change_1 = pe.Constraint(
    model.change_countries, model.car_types, rule=change_rule_1
)

In [ ]:
# Si un tipo de coche empieza a utilizarse entre dos países, y[c] debe valer 1.
def change_rule_2(model, c, t): # Determina cuando se comienza a usar el coche
    next_c = next_country[c]
    return model.x[next_c, t] - model.x[c, t] <= model.y[c]


model.change_2 = pe.Constraint(
    model.change_countries, model.car_types, rule=change_rule_2
)

## Resolver con Gurobi

## Solución óptima

In [40]:
solver = po.SolverFactory("gurobi_direct")

results = solver.solve(model, tee=True)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-10300H CPU @ 2.50GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 36 rows, 23 columns and 108 nonzeros (Min)
Model fingerprint: 0x517db828
Model has 23 linear objective coefficients
Variable types: 0 continuous, 23 integer (23 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [3e+01, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 975.0000000
Presolve time: 0.00s
Presolved: 36 rows, 23 columns, 108 nonzeros
Variable types: 0 continuous, 23 integer (23 binary)

Root relaxation: objective 8.900000e+02, 12 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node T

In [44]:
fuel_cost = sum(
    pe.value(model.unitary_cost[t, c]) * pe.value(model.x[c, t])
    for c in model.countries
    for t in model.car_types
)
switching_cost = sum(
    pe.value(model.change_cost) * pe.value(model.y[c])
    for c in model.change_countries
)

print(f"Coste de combustible: {fuel_cost:.2f} €")
print(f"Coste por cambios:   {switching_cost:.2f} €")
print(f"COSTE TOTAL MÍNIMO:  {pe.value(model.total_cost):.2f} €")

print("\nItinerario óptimo:")
for c in countries:
    car = [t for t in model.car_types if pe.value(model.x[c, t]) > 0.5]
    price = pe.value(model.unitary_cost[car, c])
    print(f"{c:12s} -> coche {car} (combustible: {price:.0f} €)")

print("\nCambios de coche:")
changes = [c for c in model.change_countries if pe.value(model.y[c]) > 0.5]
if changes:
    for c in changes:
        print(f"{c} -> {next_country[c]} (+{pe.value(model.change_cost):.0f} €)")
else:
    print("Ninguno")

assert abs(pe.value(model.total_cost) - fuel_cost - switching_cost) < 1e-6

Coste de combustible: 840.00 €
Coste por cambios:   50.00 €
COSTE TOTAL MÍNIMO:  890.00 €

Itinerario óptimo:
Spain        -> coche ['B'] (combustible: 120 €)
France       -> coche ['A'] (combustible: 210 €)
Germany      -> coche ['A'] (combustible: 180 €)
Austria      -> coche ['A'] (combustible: 110 €)
Switzerland  -> coche ['A'] (combustible: 85 €)
Italy        -> coche ['C'] (combustible: 135 €)

Cambios de coche:
Spain -> France (+25 €)
Switzerland -> Italy (+25 €)
